In [3]:
import pandas as pd
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic import BERTopic

from data import *
from ml_pipeline import *
from cluster_plot import create_interactive_topic_plot


In [31]:
data = pd.read_csv("../data/raw/csv-dump.csv")
data.head()

,global_post_id,source_id,entity_id,entity,sentiment_id,genre_id,date,view_counter,repost_counter,title,text,link
0,2_19761,2,107,РЖД,2,1,2025-04-18 6:32:09,13494,6,Главное к открытию пятницы (18.04): 👉 Значение...,Главное к открытию пятницы (18.04):\n\n👉 Значе...,https://t.me/AK47pfl/19761
1,2_19761,2,93,ЛСР,2,2,2025-04-18 6:32:09,13494,6,Главное к открытию пятницы (18.04): 👉 Значение...,Главное к открытию пятницы (18.04):\n\n👉 Значе...,https://t.me/AK47pfl/19761
2,6_13638,6,54,SOKOLOV,3,7,2025-04-19 8:04:24,4308,35,"За полгода АвтоВАЗ продал один ""суверенный эле...","За полгода АвтоВАЗ продал один ""суверенный эле...",https://t.me/Alekhin_Telega/13638
3,6_13638,6,82,Lada,3,7,2025-04-19 8:04:24,4308,35,"За полгода АвтоВАЗ продал один ""суверенный эле...","За полгода АвтоВАЗ продал один ""суверенный эле...",https://t.me/Alekhin_Telega/13638
4,7_2715,7,314,Альфа-Банк,1,2,2025-04-18 11:00:37,312502,402,Официально: мы вложили деньги в банку. В Альфа...,Официально: мы вложили деньги в банку. В Альфа...,https://t.me/AlfaBank/2715


In [5]:
data.shape

(898, 12)

In [32]:
# Создаем новый столбец с очищенным текстом
data["cleaned_text"] = data["text"].apply(clean_text)
data["cleaned_text"].iloc[0]

'Главное открытию пятницы 18 04 Значение индекса ЖиС 53 спокойствие Подробнее индексе 11 06 67 85 барр Трамп рассчитывает Вашингтон Пекин смогут выйти договоренности торговле течение ближайших трех четырех недель Трамп сообщил ближайшее время ожидает реакции Москвы инициативу прекращении огня Украине Макрон анонсировал следующий этап обсуждений урегулирования конфликта Украине пройдут Лондоне следующей неделе Россия успешно минимизировала использование недружественных валют Доля доллара евро сократилась международных резервах платежах РЖД вернулась идее запуска rde in грузовых вагонов События сегодня 1 Отсечка ЦМТ WTCM 2 ГОСА дивидендам 2024 ЛСР LSRG 3 Торги США Гонконге Германии других странах Европы проводятся связи пасхальными праздниками влияет рынки ближайшие 5 дней Доступно членам RDVPREMIUMbo Аналитика by AK47f'

In [47]:
data.to_csv("../data/interim/cleaned_text.csv", index=True, columns=["global_post_id", "cleaned_text"], index_label="raw_id")

In [48]:
data_cleaned = pd.read_csv("../data/interim/cleaned_text.csv")
data_cleaned.head()


,raw_id,global_post_id,cleaned_text
0,0,2_19761,Главное открытию пятницы 18 04 Значение индекс...
1,1,2_19761,Главное открытию пятницы 18 04 Значение индекс...
2,2,6_13638,полгода АвтоВАЗ продал суверенный электромобил...
3,3,6_13638,полгода АвтоВАЗ продал суверенный электромобил...
4,4,7_2715,Официально вложили деньги банку Альфа Банку се...


In [38]:
data_cleaned.shape

(898, 2)

In [7]:
model_name_or_path = "cointegrated/rubert-tiny2"
embedding_model = EmbeddingModel(
    method="sentence_transformer", model_name_or_path=model_name_or_path
)


In [8]:
dim_model = ReductionModel(
    method="umap", n_neighbors=15, n_components=50, min_dist=0.0, metric="euclidean"
)

In [9]:
from nltk.corpus import stopwords

stop_words = stopwords.words("russian")

In [10]:
cluster_model = ClusteringModel(
    model_name="hdbscan",
    min_cluster_size=15,
    metric="euclidean",
    cluster_selection_method="eom",
)

In [11]:
vectorizer_model = VectorizerModel(
    model_name="count", tokenizer=tokenize_ru, ngram_range=(1, 2), stop_words=stop_words
)
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)

In [12]:
representation_model = RepresentationModel(method="keybertinspired")


In [13]:
topic_model = BERTopic(
  embedding_model=embedding_model,          # Step 1 - Extract embeddings
  umap_model=dim_model,                     # Step 2 - Reduce dimensionality
  hdbscan_model=cluster_model,              # Step 3 - Cluster reduced embeddings
  vectorizer_model=vectorizer_model,        # Step 4 - Tokenize topics
  ctfidf_model=ctfidf_model,                # Step 5 - Extract topic words
  representation_model=representation_model # Step 6 - (Optional) Fine-tune topic represenations
)

In [14]:
topics, probs = topic_model.fit_transform(data['cleaned_text'])

In [15]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,180,-1_аналитики_финансирования_5 млрд_ру бизнес,"[аналитики, финансирования, 5 млрд, ру бизнес,...",[ЧАСТЬ 4 ВСЁ СМЕШАЛОСЬ ЛЮДИ КОНИ 2019 году сме...
1,0,129,0_слышит клиент_говорит клиент_сервис_клиенты,"[слышит клиент, говорит клиент, сервис, клиент...",[банковскими сусликами Хамство банковских сотр...
2,1,76,1_яндекс маркете_zo скидками_любимых брендов_с...,"[яндекс маркете, zo скидками, любимых брендов,...",[Лови большую пачку кодов крутыми горящими пре...
3,2,54,2_основатель сети_яндекс объявил_сервисов янде...,"[основатель сети, яндекс объявил, сервисов янд...",[Топ 20 компаний числу филиалов России рейтинг...
4,3,43,3_косметика это_косметика_гиалуроновой кислоты...,"[косметика это, косметика, гиалуроновой кислот...",[патчи Aexskin рассказывала ещё год назад знае...
5,4,41,4_петербургского бренда_сашина улыбка_спинке ф...,"[петербургского бренда, сашина улыбка, спинке ...",[наших любимых образов показа Саша Жаркова тот...
6,5,37,5_владислав бакальчук_владиславу бакальчуку_пр...,"[владислав бакальчук, владиславу бакальчуку, п...",[Новости 9 00 МСК Дональд Трамп вслед госсекре...
7,6,33,6_снижения ставки_меры поддержки_роста цен_инв...,"[снижения ставки, меры поддержки, роста цен, и...",[Сибирь Итоги недели Губернатор Республики Бур...
8,7,31,7_дебетовой карты_дебетовую карту_обслуживание...,"[дебетовой карты, дебетовую карту, обслуживани...",[Получаем сертификат 800р OZON Золотое яблоко ...
9,8,31,8_aiexress icick_чистки окон_icick sho_окон ян...,"[aiexress icick, чистки окон, icick sho, окон ...",[Смартфон Xioi 13T 8 256 ГБ 6 67 AMOLED 2712x1...


In [16]:
new_topic_names = create_topic_names(topic_model, embedding_model, stop_words)


In [49]:
new_topic_names

{0: 'рядовых банковских сотрудников',
 1: 'p25vayjejgfvf скидка 50',
 2: 'основатель сети магазинов',
 3: 'доказывает качественная косметика',
 4: 'фирменная сашина улыбка',
 5: 'бизнес германова ответила',
 6: 'заемщиков правительство планирует',
 7: 'оформляем дебетовую карту',
 8: 'erid 5jcerenx12ojveypja3q6 450',
 9: 'мл 104 бальзам',
 10: 'аналитиков считаем индекс',
 11: 'кредита займа сайте',
 12: 'яндекс маркете доставка',
 13: 'нлмк нижнекамскнефтехим ozon',
 14: 'титульным спонсором российской',
 15: 'помощью сервиса онлайн',
 16: 'электрический стеклоочиститель puruiki',
 17: 'блогеров маркетплейс начнёт',
 18: 'пользуется благами банков',
 19: 'массажер пистолет устройство',
 20: 'зелени заказать яндекс'}

In [25]:
dim_model.embedding_.shape

(898, 50)

In [54]:
type(topics)

list

In [59]:
dim_model.embedding_.shape

(898, 2)

In [21]:
# Сохраняем embedding в файл
import numpy as np
np.save('../data/external/embeddings.npy', dim_model.embedding_)
print(f"Embedding сохранен в файл embeddings.npy, размерность: {dim_model.embedding_.shape}")
dim_model.embedding_


Embedding сохранен в файл embeddings.npy, размерность: (898, 50)


array([[9.925478  , 0.3585881 , 0.22108005, ..., 6.2596517 , 5.8263636 ,
        5.195323  ],
       [9.924407  , 0.35608202, 0.21941471, ..., 6.2606544 , 5.8244677 ,
        5.195173  ],
       [9.897067  , 0.3047334 , 0.6505235 , ..., 6.430326  , 5.617011  ,
        5.191135  ],
       ...,
       [9.73876   , 0.40153596, 1.4630415 , ..., 6.545523  , 5.5043035 ,
        5.1302557 ],
       [9.64246   , 0.53059274, 1.2407649 , ..., 6.2609096 , 5.857716  ,
        5.2037516 ],
       [9.76827   , 0.52952045, 1.1494702 , ..., 6.539642  , 4.931425  ,
        5.069058  ]], dtype=float32)

In [23]:
dim_model_ = np.load('../data/external/embeddings.npy')
dim_model_.shape



(898, 50)

In [25]:

from utils_ import save_np_tesors, load_np_tesors
save_np_tesors(np.array(topics), '../data/external/topics.npy')
topics = load_np_tesors('../data/external/topics.npy')
topics.shape





(898,)

In [52]:
# Создаем интерактивный график кластеров
fig = create_interactive_topic_plot(
    dim_model=dim_model,  # Модель снижения размерности
    topics=topics,  # Метки кластеров
    data=data,  # Датафрейм с текстами
    new_topic_names=new_topic_names,  # Словарь с названиями топиков
    dimensions=3
)

# Отображаем график
show_interactive_topic_plot(fig, fullscreen=True)

TypeError: create_interactive_topic_plot() got an unexpected keyword argument 'dimensions'

In [ ]:
dim2d = 